NOTE: THIS DOES NOT INCORPORATE TRANSACTION FEES!

In [1]:
import pandas as pd
import numpy as np
from scipy.optimize import minimize
import warnings
warnings.filterwarnings('ignore')

def bsv_utility(theta, r_bench, f_ret, gamma=5):
    """
    CRRA Utility Objective Function for BSV.
    r_bench: Array of equal-weighted benchmark returns
    f_ret: Matrix of characteristic-weighted returns (T x K)
    theta: Characteristic coefficients (K,)
    gamma: Coefficient of relative risk aversion
    """
    # Calculate portfolio return given theta
    r_p = r_bench + np.dot(f_ret, theta)
    
    # Wealth must be positive for CRRA utility
    wealth = 1 + r_p
    if np.any(wealth <= 0):
        return 1e10  # Heavy penalty for bankruptcy
        
    if gamma == 1:
        u = np.log(wealth)
    else:
        u = (wealth ** (1 - gamma)) / (1 - gamma)
        
    # We want to MAXIMIZE utility, so we MINIMIZE negative utility
    return -np.mean(u)

def backtest_bsv(df, test_start_date='2021-10-31', test_end_date='2024-04-30', 
                 lookback_window=180, gamma=5):
    print("="*60)
    print("STARTING BSV (2009) BENCHMARK OPTIMIZATION")
    print("="*60)
    
    df = df.copy()
    features = ['bm', 'mve', 'mom12m']
        
    # Pre-calculate Equal-Weighted (Benchmark) Returns and Factor Returns for speed
    print("Pre-computing characteristic factor returns...")
    
    # Filter out rows with no forward return
    valid_df = df.dropna(subset=['ret_fwd_1'])
    
    monthly_data = []
    for date, group in valid_df.groupby('datadate'):
        N = len(group)
        if N == 0:
            continue
            
        r_bench = group['ret_fwd_1'].mean()
        
        # Characteristic portfolio returns: f_t = (1/N) * sum(x_{i,t} * r_{i,t+1})
        f_bm = (group['bm'] * group['ret_fwd_1']).mean()
        f_mve = (group['mve'] * group['ret_fwd_1']).mean()
        f_mom = (group['mom12m'] * group['ret_fwd_1']).mean()
        
        monthly_data.append({
            'datadate': date,
            'r_bench': r_bench,
            'f_bm': f_bm,
            'f_mve': f_mve,
            'f_mom': f_mom,
            'N': N
        })
        
    factors_df = pd.DataFrame(monthly_data).set_index('datadate').sort_index()
    
    # Prepare Dates
    all_dates = factors_df.index.tolist()
    test_start_dt = pd.to_datetime(test_start_date)
    test_end_dt = pd.to_datetime(test_end_date)
    
    try:
        test_start_idx = all_dates.index(test_start_dt)
        test_end_idx = all_dates.index(test_end_dt)
    except ValueError as e:
        raise ValueError(f"Date not found in valid dates: {e}")

    oos_returns = []
    oos_dates = []
    thetas = []

    print(f"Running rolling optimizations ({lookback_window}-month window)...")
    for t in range(test_start_idx, test_end_idx + 1):
        current_date = all_dates[t]
        
        if t < lookback_window:
            print(f"  Skipping {current_date.strftime('%Y-%m')}: insufficient history.")
            continue
            
        # Get training window (strictly prior to current date)
        train_window = factors_df.iloc[t - lookback_window : t]
        
        r_bench_train = train_window['r_bench'].values
        f_ret_train = train_window[['f_bm', 'f_mve', 'f_mom']].values
        
        # Optimize theta
        theta_0 = np.zeros(len(features)) # Start exactly at the equal-weighted benchmark
        res = minimize(bsv_utility, theta_0, args=(r_bench_train, f_ret_train, gamma), 
                       method='BFGS')
        
        theta_opt = res.x
        
        # Calculate Out-of-Sample Return for current date
        current_data = factors_df.iloc[t]
        r_bench_oos = current_data['r_bench']
        f_ret_oos = current_data[['f_bm', 'f_mve', 'f_mom']].values
        
        r_portfolio_oos = r_bench_oos + np.dot(f_ret_oos, theta_opt)
        
        oos_returns.append(r_portfolio_oos)
        oos_dates.append(current_date)
        thetas.append(theta_opt)
        
    # Compile Results
    results_df = pd.DataFrame({
        'date': oos_dates,
        'bsv_return': oos_returns
    })
    
    theta_df = pd.DataFrame(thetas, columns=features, index=oos_dates)
    
    # Calculate Metrics
    mean_ret = np.mean(oos_returns)
    var_ret = np.var(oos_returns, ddof=1)
    
    # Standard monthly Sharpe (assuming risk-free rate is absorbed/0 for simplicity)
    sr = mean_ret / np.sqrt(var_ret) if var_ret > 0 else 0
    
    # Annualized
    ann_mean = mean_ret * 12
    ann_vol = np.sqrt(var_ret * 12)
    ann_sr = ann_mean / ann_vol if ann_vol > 0 else 0

    print("\n" + "="*60)
    print(f"BSV (2009) RESULTS: {test_start_dt.strftime('%Y-%m')} to {test_end_dt.strftime('%Y-%m')}")
    print("="*60)
    print(f"Average Monthly Return: {mean_ret:.6f} ({ann_mean*100:.2f}% Annually)")
    print(f"Monthly Variance      : {var_ret:.6f}")
    print(f"Monthly Sharpe Ratio  : {sr:.4f}")
    print(f"Annualized Sharpe     : {ann_sr:.4f}")
    print(f"Average Theta Values  : bm={theta_df['bm'].mean():.4f}, mve={theta_df['mve'].mean():.4f}, mom12m={theta_df['mom12m'].mean():.4f}")
    print("="*60)
    
    return results_df, theta_df

In [3]:
# ==========================================
# Example Execution
# ==========================================
# Ensure df is loaded exactly as you did in your POET code
df = pd.read_csv('../green cleaned.csv', dtype={'ncusip': 'string'})
df['datadate'] = pd.to_datetime(df['datadate'])
df['ret_fwd_1'] = df.groupby('permno')['ret_excess'].shift(-1)

In [12]:
bsv_returns, bsv_thetas = backtest_bsv(
    df=df,
    test_start_date='2021-10-31',
    test_end_date='2024-04-30',
    lookback_window=180,
    gamma=5 # standard risk aversion parameter from BSV
)

STARTING BSV (2009) BENCHMARK OPTIMIZATION
Pre-computing characteristic factor returns...
Running rolling optimizations (180-month window)...

BSV (2009) RESULTS: 2021-10 to 2024-04
Average Monthly Return: -0.001508 (-1.81% Annually)
Monthly Variance      : 0.005845
Monthly Sharpe Ratio  : -0.0197
Annualized Sharpe     : -0.0683
Average Theta Values  : bm=-3.0794, mve=-1.7725, mom12m=0.3532


In [13]:
bsv_thetas

,bm,mve,mom12m
2021-10-31,-4.761821,-2.353571,-0.141718
2021-11-30,-4.758501,-2.144728,-0.222053
2021-12-31,-5.007047,-2.369097,-0.286224
2022-01-31,-3.469662,-1.514939,0.036273
2022-02-28,-3.390611,-1.773742,0.241775
2022-03-31,-3.404222,-1.681792,0.178421
2022-04-30,-3.299028,-1.879620,0.379674
2022-05-31,-2.910016,-1.748843,0.575133
2022-06-30,-2.956884,-1.625990,0.524977
2022-07-31,-3.011248,-1.839553,0.565655


In [14]:
bsv_returns

,date,bsv_return
0,2021-10-31,-0.001961
1,2021-11-30,0.091429
2,2021-12-31,-0.222121
3,2022-01-31,-0.027703
4,2022-02-28,0.032423
5,2022-03-31,-0.068494
6,2022-04-30,-0.049267
7,2022-05-31,-0.068061
8,2022-06-30,0.143240
9,2022-07-31,-0.051526
